# YouTube data quality check

**Input:**
- `data/interim/youtube.csv` — monthly YouTube Content ID reports, 01-2024 → 09-2026.
- `data/reference/releases.csv` — release list of the catalogue (artist, track, ISRC, UPC, release date).

**Goal:** find data problems *before* cleaning. This notebook only looks at the data and changes nothing; the fixes go to `src/clean_data.py`.

A summary table is at the end.

In [16]:
import pandas as pd

# upc is read as text: it is a code, not a number
data = pd.read_csv(
    "../data/interim/youtube.csv",
    parse_dates=["report_month"],
    dtype={"upc": str},
)
releases = pd.read_csv("../data/reference/releases.csv", dtype={"upc": str})

## 1. Overview

**Check:** table size and first rows.

In [ ]:
# Size of the table (rows, columns) and a first look at the data
print(data.shape)
data.head()

(34470, 8)


**Found:** 34,470 rows, 8 columns.

**Decision:** no action needed.

## 2. Column types

In [18]:
data.dtypes

artist                     str
track                      str
isrc                       str
upc                        str
revenue                float64
video_url                  str
payment_type               str
report_month    datetime64[us]
dtype: object

**Found:** `revenue` is a float, `report_month` is a datetime, the other columns are text, including the codes `isrc` (track) and `upc` (release).

**Decision:** no action needed.

## 3. Missing values

**Check:** are there empty cells?

In [19]:
data.isna().sum()

artist             0
track              0
isrc            5881
upc                0
revenue            0
video_url          0
payment_type       0
report_month       0
dtype: int64

**Found:** `isrc` is empty in 5,881 rows; all other columns have no missing values.

**Decision:** in `clean_data.py`, fill the missing `isrc` from the release list `data/reference/releases.csv` (matched by `artist` + `track`).

## 4. Negative and zero revenue

**Check:** royalties should not be negative; zero-revenue rows add nothing to the analysis.

In [20]:
print("Negative revenue:", (data["revenue"] < 0).sum())
print("Zero revenue:", (data["revenue"] == 0).sum())

Negative revenue: 0
Zero revenue: 0


**Found:** no negative and no zero values.

**Decision:** no action needed.

## 5. Artists, tracks and codes vs release list

**Check:** every row matches the release list: rows with ISRC by `artist` + `track` + `isrc` + `upc`, rows without ISRC (section 3) by `artist` + `track` + `upc`. A row with a different spelling or a wrong code would show up as "not found".
- **ISRC** — code of a recording (track); **UPC** — code of a release (single, album).

In [21]:
data[["artist", "track", "payment_type"]].nunique()

artist           6
track           18
payment_type     2
dtype: int64

In [23]:
data["payment_type"].value_counts()

payment_type
red    22205
ads    12265
Name: count, dtype: int64

In [24]:
# Compare every row with the release list:
# rows with ISRC by artist + track + isrc + upc, rows without ISRC by artist + track + upc
full_key = ["artist", "track", "isrc", "upc"]
short_key = ["artist", "track", "upc"]

has_isrc = data["isrc"].notna()
with_isrc = data.loc[has_isrc, full_key].merge(releases[full_key], how="left", indicator=True)
without_isrc = data.loc[~has_isrc, short_key].merge(releases[short_key], how="left", indicator=True)

print("Rows with ISRC:", len(with_isrc), "| not found:", (with_isrc["_merge"] == "left_only").sum())
print("Rows without ISRC:", len(without_isrc), "| not found:", (without_isrc["_merge"] == "left_only").sum())

Rows with ISRC: 28589 | not found: 0
Rows without ISRC: 5881 | not found: 0


**Found:**
- 6 artists, 18 tracks, 2 payment types: `ads` (advertising) and `red` (YouTube Premium).
- All 28,589 rows with ISRC match the release list on `artist` + `track` + `isrc` + `upc`.
- All 5,881 rows without ISRC match it on `artist` + `track` + `upc`.

**Decision:**
- No action needed; `upc` is kept as in the reports (same form as in the release list).
- No re-check is needed after the missing `isrc` is filled (section 3): each code is taken from the release list itself, so it matches by definition, and every row without ISRC already finds its track there.
- `isrc` is the track key in the analysis (industry standard, does not depend on spelling). It is used only inside the project and is not shown in public outputs.

## 6. Video URL format

**Check:** every `video_url` is a standard YouTube link: `https://www.youtube.com/watch?v=` + an 11-character video ID.

In [ ]:
# Standard link = fixed prefix + 11 characters (letters, digits, "_" or "-")
pattern = r"^https://www\.youtube\.com/watch\?v=[A-Za-z0-9_-]{11}$"
is_standard = data["video_url"].str.match(pattern)
print("Non-standard URLs:", (~is_standard).sum())
data[~is_standard]

**Found:** 2 rows (one month, one track, tiny revenue) have a non-standard link of the form `watch?v=deleted-...` — the video was deleted, so YouTube replaced its ID.

**Decision:** keep both rows — the revenue was really paid. In `clean_data.py` replace their `video_url` with `deleted`. `video_url` is kept only as a reference to open a video when needed.

## 7. Rows and revenue per month

**Check:** does the number of rows or the revenue jump unexpectedly from month to month?

In [ ]:
# Rows and total revenue in each monthly report
monthly = data.groupby("report_month").agg(
    rows=("revenue", "size"),
    revenue=("revenue", "sum"),
)
monthly

**Found:**
- Revenue falls noticeably from 04/05-2025. This is not a data error: the decline will be analysed in a separate notebook.
- From 06-2026 there are only **6–7 rows per month**, and **09-2026** has almost zero revenue.

**Decision:**
- 2025 decline → real business dynamics, kept as is; it will be analysed separately.
- 06–09 2026 → real data, kept as is. The contract between the label and the distributor was terminated, and the catalogue was gradually taken down to be re-released elsewhere. In the analysis notebooks these months will be treated as a wind-down period: marked on charts and not compared directly with earlier months.

## Summary

| # | Check | Found | Decision |
|---|---|---|---|
| 1 | Overview | 34,470 rows, 8 columns | OK |
| 2 | Column types | as expected | no action |
| 3 | Missing values | `isrc` empty in 5,881 rows | fill from the release list `releases.csv` |
| 4 | Negative / zero revenue | none | no action |
| 5 | Artists, tracks, codes vs release list | 6 / 18 / 2 values; every row (with and without ISRC) matches the release list | `isrc` = track key; `upc` as in the reports; no re-check after filling |
| 6 | Video URL format | 2 deleted-video links | keep rows, set `video_url` to `deleted` |
| 7 | Rows per month | revenue decline from 04/05-2025; 06–09 2026 catalogue wind-down | keep both; analyse the decline separately; treat 06–09 2026 as wind-down period |

**Next step:** `02_dsp_data_quality.ipynb`.